In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import numpy as np

BASE_PATH = "/content/drive/My Drive/Datasets"


In [4]:
sod_df = pd.read_csv(f"{BASE_PATH}/Inventory_Snapshot_SOD.csv")
eod_df = pd.read_csv(f"{BASE_PATH}/Inventory_Snapshot_EOD.csv")
recv_df = pd.read_csv(f"{BASE_PATH}/Receiving_Transactions.csv")
ship_df = pd.read_csv(f"{BASE_PATH}/Shipping_Transactions.csv")


Inventory SOD, sum of on-hand quantity of the item

In [5]:
sod_agg = (
    sod_df
    .groupby(["item_id", "warehouse_id"], as_index=False)
    .agg(sod_qty=("quantity_on_hand", "sum"))
)


Sum of Receiving quantities of Item

In [6]:
recv_agg = (
    recv_df
    .groupby(["item_id", "warehouse_id"], as_index=False)
    .agg(total_received=("received_qty", "sum"))
)


Sum of Shipping Quantities of Item

In [8]:
ship_agg = (
    ship_df[ship_df["order_status"] == "SHIPPED"]
    .groupby(["item_id", "warehouse_id"], as_index=False)
    .agg(total_shipped=("ship_qty", "sum"))
)


Sum of Inventory of Item at EOD

In [9]:
eod_agg = (
    eod_df
    .groupby(["item_id", "warehouse_id"], as_index=False)
    .agg(eod_qty=("quantity_on_hand", "sum"))
)

Logic

In [10]:
recon_df = (
    sod_agg
    .merge(recv_agg, on=["item_id", "warehouse_id"], how="left")
    .merge(ship_agg, on=["item_id", "warehouse_id"], how="left")
    .merge(eod_agg, on=["item_id", "warehouse_id"], how="left")
)

recon_df.fillna(0, inplace=True)

recon_df["expected_eod_qty"] = (
    recon_df["sod_qty"]
    + recon_df["total_received"]
    - recon_df["total_shipped"]
)

recon_df["variance"] = recon_df["eod_qty"] - recon_df["expected_eod_qty"]


In [11]:
recon_df["anomaly_flag"] = np.where(
    recon_df["variance"] != 0,
    "Y",
    "N"
)


In [12]:
def classify_severity(row):
    if row["variance"] == 0:
        return "NONE"
    elif abs(row["variance"]) <= 5:
        return "LOW"
    elif abs(row["variance"]) <= 20:
        return "MEDIUM"
    else:
        return "HIGH"

recon_df["anomaly_severity"] = recon_df.apply(classify_severity, axis=1)


In [13]:
anomaly_report = recon_df[
    recon_df["anomaly_flag"] == "Y"
].sort_values(
    by="variance",
    key=lambda x: abs(x),
    ascending=False
)


In [14]:
anomaly_report.to_csv(
    f"{BASE_PATH}/Inventory_Anomaly_Report.csv",
    index=False
)
